# 집값 예측 — 설정값 찾기

모델은 골랐습니다. 남은 일은 **손잡이(하이퍼파라미터)** 를 고르는 일입니다.  
전부 다 해보기(**GridSearchCV**)와 몇 번만 뽑아 보기(**RandomizedSearchCV**)를 비교합니다.


In [9]:
pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   - -------------------------------------- 2.4/48.9 MB 11.9 MB/s eta 0:00:04
   --- ------------------------------------ 4.5/48.9 MB 11.0 MB/s eta 0:00:05
   ----- ---------------------------------- 6.6/48.9 MB 10.7 MB/s eta 0:00:04
   ------ --------------------------------- 8.4/48.9 MB 10.5 MB/s eta 0:00:04
   -------- ------------------------------- 11.0/48.9 MB 10.7 MB/s eta 0:00:04
   ---------- ----------------------------- 13.4/48.9 MB 10.9 MB/s eta 0:00:04
   ------------ --------------------------- 15.7/48.9 MB 10.9 MB/s eta 0:00:04
   -------------- ------------------------- 17.8/48.9 MB 11.0 MB/s eta 0:00:03
   ---------------- ----------------------- 19.7/48.9 MB 10.8 MB/s eta 0:00:03
   ----------------- ---------------------- 21.8/48.9 MB 10.7 MB/s eta 0:00:03
   ------------------- -------------------- 23.9/48.9 MB 10.6 MB/s eta 0:00:03
   --------------------- ------------------ 26.0/48.9 MB 10.6 MB/

In [10]:
# 집값 표·모델·탐색 도구 불러오기
import numpy as np
import pandas as pd

# GridSearchCV = 설정 조합을 전부 시험
# RandomizedSearchCV = 설정 조합을 일부만 무작위 시험
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score

# xgboost가 없으면 그 모델만 건너뜀
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    XGBRegressor = None
    HAS_XGB = False
    print("xgboost 미설치 — XGBRegressor는 스킵합니다. (pip install xgboost)")

In [11]:
# 캘리포니아 집값 표를 DataFrame으로 만들기
# MedHouseVal = 맞출 집값(중간값)
housing = fetch_california_housing(as_frame=True)
df = housing.frame

print(df.head())
print("컬럼:", df.columns.tolist())
print("행 개수:", len(df))

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  MedHouseVal  
0    -122.23        4.526  
1    -122.22        3.585  
2    -122.24        3.521  
3    -122.25        3.413  
4    -122.25        3.422  
컬럼: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'MedHouseVal']
행 개수: 20640


## 나누기 & 스케일

연습/시험으로 나눈 뒤, 스케일은 **연습에만 fit** 합니다.  
시험 분포를 미리 보면 **데이터 누수**입니다 (점수가 가짜로 좋아짐).


In [12]:
# 연습 80% / 시험 20%로 나눔. 스케일러는 연습에만 fit (시험 미리 보면 부정행위)
# X = 집값 뺀 나머지 칸(문제), y = 집값(정답)
X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# StandardScaler = 칸마다 평균 0·퍼짐 1로 맞춤
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit은 연습만
X_test_scaled = scaler.transform(X_test)        # 시험은 변환만 적용

print("X_train shape:", X_train_scaled.shape)
print("X_test shape:", X_test_scaled.shape)

X_train shape: (16512, 8)
X_test shape: (4128, 8)


In [13]:
# 모델마다 돌려볼 설정값 목록
# 주의: 모든 조합을 다 돌려서 시간이 길 수 있음
# params = 모델 손잡이(깊이, 나무 수 등) 후보
model_configs = {
    "Ridge": {
        "model": Ridge(),
        "params": {"alpha": [0.1, 1.0, 10.0, 100.0]},
    },
    "Lasso": {
        "model": Lasso(max_iter=5000),
        "params": {"alpha": [0.001, 0.01, 0.1, 1.0]},
    },
    "DecisionTree": {
        "model": DecisionTreeRegressor(random_state=42),
        "params": {
            "max_depth": [3, 5, 7, 10, None],
            "min_samples_split": [2, 5, 10],
        },
    },
    "RandomForest": {
        "model": RandomForestRegressor(random_state=42),
        "params": {
            "n_estimators": [50, 100, 200],
            "max_depth": [5, 10, 15, None],
            "min_samples_split": [2, 5],
        },
    },
    "GradientBoosting": {
        "model": GradientBoostingRegressor(random_state=42),
        "params": {
            "n_estimators": [50, 100, 200],
            "learning_rate": [0.05, 0.1, 0.2],
            "max_depth": [3, 5, 7],
        },
    },
}

if HAS_XGB:
    model_configs["XGB"] = {
        "model": XGBRegressor(
            random_state=42,
            objective="reg:squarederror",
            n_jobs=-1,
        ),
        "params": {
            "n_estimators": [50, 100, 200],
            "learning_rate": [0.05, 0.1, 0.2],
            "max_depth": [3, 5, 7],
        },
    }

print("모델 목록:", list(model_configs.keys()))

모델 목록: ['Ridge', 'Lasso', 'DecisionTree', 'RandomForest', 'GradientBoosting', 'XGB']


## GridSearchCV

적어 둔 설정 조합을 **전부** 시험합니다. 정확하지만 시간이 오래 걸릴 수 있습니다.


In [6]:
# 모든 설정 조합을 하나씩 시험 (시간 걸릴 수 있음)
grid_rows = []

for name, cfg in model_configs.items():
    print(f"\n=== GridSearchCV: {name} ===")
    # GridSearchCV = 손잡이 후보를 격자처럼 전부 대입해 최고 조합 찾기
    # cv=5 = 연습 데이터를 5등분해서 번갈아 채점
    # scoring = 채점 방식(여기선 오제곱 오차의 음수; 클수록 좋음)
    grid = GridSearchCV(
        estimator=cfg["model"],
        param_grid=cfg["params"],
        scoring="neg_mean_squared_error",
        cv=5,
        n_jobs=-1,
    )
    # fit = 격자 탐색으로 최고 손잡이 고르기
    grid.fit(X_train_scaled, y_train)

    # best_estimator_ = 고른 최고 모델로 시험 점수 예측
    y_pred = grid.best_estimator_.predict(X_test_scaled)
    test_r2 = r2_score(y_test, y_pred)

    print("best_params_:", grid.best_params_)
    print(f"test_r2: {test_r2:.4f}")

    grid_rows.append(
        {
            "model": name,
            "best_params": grid.best_params_,
            "test_r2": test_r2,
        }
    )

# 모델별 시험 R²를 표로 모아 높은 순 정렬
grid_df = pd.DataFrame(grid_rows).sort_values("test_r2", ascending=False)
print("\n--- GridSearchCV test_r2 표 ---")
print(grid_df.to_string(index=False))


=== GridSearchCV: Ridge ===
best_params_: {'alpha': 0.1}
test_r2: 0.5758

=== GridSearchCV: Lasso ===
best_params_: {'alpha': 0.001}
test_r2: 0.5769

=== GridSearchCV: DecisionTree ===
best_params_: {'max_depth': 10, 'min_samples_split': 10}
test_r2: 0.6822

=== GridSearchCV: RandomForest ===
best_params_: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
test_r2: 0.8060

=== GridSearchCV: GradientBoosting ===
best_params_: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200}
test_r2: 0.8417

--- GridSearchCV test_r2 표 ---
           model                                                      best_params  test_r2
GradientBoosting      {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200} 0.841703
    RandomForest {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200} 0.806007
    DecisionTree                       {'max_depth': 10, 'min_samples_split': 10} 0.682166
           Lasso                                                 {'alpha': 0.001} 0.

## RandomizedSearchCV

설정 중 **몇 번만 무작위**로 뽑습니다. 어디서 괜찮은지 빨리 감을 잡을 때 씁니다.  
실무: 넓게 Random → 좁혀 Grid.


In [7]:
# 같은 설정 목록에서 무작위로 최대 10개만 뽑아 비교
# 넓게 빨리 훑기. 조합이 10개보다 적으면 그만큼만 돌림
from functools import reduce
import operator

rand_rows = []

for name, cfg in model_configs.items():
    print(f"\n=== RandomizedSearchCV: {name} ===")
    # 손잡이 조합이 모두 몇 개인지 곱해서 셈
    n_combos = reduce(operator.mul, (len(v) for v in cfg["params"].values()), 1)
    n_iter = min(10, n_combos)
    # RandomizedSearchCV = 전부 말고 n_iter개만 뽑아 시험
    rand = RandomizedSearchCV(
        estimator=cfg["model"],
        param_distributions=cfg["params"],
        n_iter=n_iter,
        scoring="neg_mean_squared_error",
        cv=5,
        n_jobs=-1,
        random_state=42,
    )
    rand.fit(X_train_scaled, y_train)

    y_pred = rand.best_estimator_.predict(X_test_scaled)
    test_r2 = r2_score(y_test, y_pred)

    print("best_params_:", rand.best_params_)
    print(f"test_r2: {test_r2:.4f}")

    rand_rows.append(
        {
            "model": name,
            "best_params": rand.best_params_,
            "test_r2": test_r2,
        }
    )

rand_df = pd.DataFrame(rand_rows).sort_values("test_r2", ascending=False)
print("\n--- RandomizedSearchCV test_r2 표 ---")
print(rand_df.to_string(index=False))


=== RandomizedSearchCV: Ridge ===
best_params_: {'alpha': 0.1}
test_r2: 0.5758

=== RandomizedSearchCV: Lasso ===
best_params_: {'alpha': 0.001}
test_r2: 0.5769

=== RandomizedSearchCV: DecisionTree ===
best_params_: {'min_samples_split': 10, 'max_depth': 10}
test_r2: 0.6822

=== RandomizedSearchCV: RandomForest ===
best_params_: {'n_estimators': 100, 'min_samples_split': 2, 'max_depth': 15}
test_r2: 0.8006

=== RandomizedSearchCV: GradientBoosting ===
best_params_: {'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.1}
test_r2: 0.8417

--- RandomizedSearchCV test_r2 표 ---
           model                                                    best_params  test_r2
GradientBoosting    {'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.1} 0.841703
    RandomForest {'n_estimators': 100, 'min_samples_split': 2, 'max_depth': 15} 0.800576
    DecisionTree                     {'min_samples_split': 10, 'max_depth': 10} 0.682166
           Lasso                                          

In [8]:
# 한 줄 요약
print("실무 흐름: RandomizedSearchCV로 넓게 훑고 → GridSearchCV로 좁혀 전수 탐색")
print("scoring=neg_mean_squared_error (MSE는 작을수록 좋은데, sklearn은 클수록 좋음)")
print("StandardScaler는 train에만 fit (누수 방지)")
if not HAS_XGB:
    print("참고: xgboost 미설치로 XGB는 이번 실행에서 제외됨")

실무 흐름: RandomizedSearchCV로 넓게 훑고 → GridSearchCV로 좁혀 전수 탐색
scoring=neg_mean_squared_error (MSE는 작을수록 좋은데, sklearn은 클수록 좋음)
StandardScaler는 train에만 fit (누수 방지)
참고: xgboost 미설치로 XGB는 이번 실행에서 제외됨
